In [1]:
from dotenv import load_dotenv
import os
from google import genai
import requests

# Load environment variables from your .env file
load_dotenv()

# Initialize the Gemini client using your environment variable
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Test a simple prompt to verify everything works flawlessly
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Hello Gemini! Give me a quick confirmation that you are online.'
)

print(response.text)

Hello! Yes, I am online and ready to help.


In [2]:
def llm(prompt):
    # Fixed the spelling to 'response'
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config={'system_instruction': 'You are a helpful course assistant.'}
    )
    # Added the return statement so the function actually outputs the text
    return response.text

In [3]:
llm("Hey, what's up?")

'Hey there! Not much, just ready to assist you.\n\nWhat can I help you with today regarding your course? Do you have a question, need help finding something, or want to discuss a topic?'

In [4]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [5]:
answer = llm(prompt)
print(answer)

NameError: name 'prompt' is not defined

In [6]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [7]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [8]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [9]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [10]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [11]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [12]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [13]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [14]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [15]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [16]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [18]:
response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
)

In [21]:
print(response.text)

Yes, you can join the course now. You can start learning whenever you want, as the videos and GitHub materials are available.

However, if you wish to receive a certificate, you must complete the course with a "live" cohort. This means submitting your project while submissions are open and being able to participate in the peer-review process (reviewing 3 capstone projects), which is only possible when the course is actively running. Certificates are not awarded for self-paced completion outside of these "live" cohort activities.


In [23]:
print(response.text)

Yes, you can join the course now. You can start learning whenever you want, as the videos and GitHub materials are available.

However, if you wish to receive a certificate, you must complete the course with a "live" cohort. This means submitting your project while submissions are open and being able to participate in the peer-review process (reviewing 3 capstone projects), which is only possible when the course is actively running. Certificates are not awarded for self-paced completion outside of these "live" cohort activities.


In [26]:
print(response.usage_metadata)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=107 candidates_tokens_details=None prompt_token_count=514 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=514
)] thoughts_token_count=1121 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=1742 traffic_type=None


## Token Cost 


In [27]:
def calculate_gemini_cost(usage):
    # Extract token counts
    input_tokens = usage.prompt_token_count
    
    # Crucial step: output includes both the final text and the thinking tokens
    output_tokens = usage.candidates_token_count + (usage.thoughts_token_count or 0)
    
    # Rates per 1 million tokens
    input_rate_per_million = 0.075
    output_rate_per_million = 0.30
    
    # Calculate cost
    input_cost = (input_tokens / 1_000_000) * input_rate_per_million
    output_cost = (output_tokens / 1_000_000) * output_rate_per_million
    
    total_cost = input_cost + output_cost
    return total_cost

# Pass your metadata into the function
cost = calculate_gemini_cost(response.usage_metadata)
print(f"Total query cost: ${cost:.6f}")

Total query cost: $0.000407


In [35]:
from google import genai
from google.genai import types

def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    # 1. We bundle the system instructions into Gemini's configuration object
    config = types.GenerateContentConfig(
        system_instruction=instructions
    )
    
    # 2. Call the Gemini client to generate the content
    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=config
    )
    
    # 3. Cleanly extract just the text answer
    return response.text

In [ ]:
def rag(query, model="gemini-2.5-flash"):
    search_results = index.search(
        query,
        boost_dict={"question": 2.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
        num_results=5
    )
    prompt = build_prompt(query, search_results)
    
    # Use the correct variable name you defined in cell [39]
    answer = llm(system_rules, prompt, model=model)
    return answer

In [39]:
system_rules = "You are an expert Data Engineer teaching a beginner."
question = "What is the difference between a Spark DataFrame and an RDD?"

answer = llm(system_rules, question)
print(answer)

That's an excellent question for someone starting out with Spark! It gets right to the heart of how Spark handles data.

Think of it like this:

*   A **Spark RDD** is like having a **big box of loose LEGO bricks.** You know they are all LEGOs, but you don't know what kind of structure they form until you start putting them together yourself.
*   A **Spark DataFrame** is like having a **pre-built LEGO set with instructions and clearly labeled sections.** You know exactly what each piece is for, where it goes, and how it all fits together to form a specific model.

Let's break down the technical differences:

---

### What is an RDD (Resilient Distributed Dataset)?

The RDD (Resilient Distributed Dataset) is the **fundamental, low-level data structure** in Spark. It was the primary API when Spark first launched.

1.  **Low-Level Abstraction:** It represents an immutable, partitioned collection of elements that can be operated on in parallel across a cluster.
2.  **Schema-less / Unstruct